# 01. Data Understanding

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Goal of this notebook:** look at the raw shipment file before we change anything, and decide whether it is good enough to train a failure-prediction model on.

**Who this is for:** a healthcare logistics coordinator needs to know which cold-chain shipment will fail *before* it arrives. A model can only help them if the data behind it behaves like real shipments.


### What we check
1. What is in the file (size, columns, types)
2. Is anything missing or duplicated
3. What the numbers and categories look like
4. Do the values make physical sense
5. Do the columns relate to each other the way real shipments would
6. Is there an outcome to predict
7. What we found and what happens next

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_PATH = ROOT / "data" / "raw" / "global_supply_chain_pharma.csv"

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig


raw = pd.read_csv(RAW_PATH)
print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns from {RAW_PATH.relative_to(ROOT)}")

Loaded 5,000 rows and 15 columns from data\raw\global_supply_chain_pharma.csv


## 1. What is in the file

One row is one shipment. Here are the first five.

In [2]:
raw.head()

,Shipment_ID,Place_of_Origin,Place_of_Destination,Distance_km,Transport_Mode,Lead_Time_Days,Transit_Duration_Hours,Stock_Volume,Geopolitical_Risk_Score,Fuel_Price_Index,Carrier_Reliability_Score,Buffer_of_Package,Storage_standard,Cargo_Value_USD,Hospital_Urgency_Tier
0,SC-10000,Singapore Pharmaceutical Logistics Depot (synt...,Los Angeles Medical Centre Depot (synthetic),5930.83,Multi-Modal Air-to-Land,41.39,34.39,6241,5.0,2.43,0.865,15.63,2°C to 8°C,28618.15,Routine Restock
1,SC-10001,Singapore Pharmaceutical Logistics Depot (synt...,Shanghai Medical Centre Depot (synthetic),14285.36,Multi-Modal Air-to-Land,40.92,59.13,280,7.5,2.30,0.592,8.08,2°C to 8°C,56468.28,Elevated Procedure
2,SC-10002,Rotterdam Pharmaceutical Logistics Depot (synt...,Los Angeles Medical Centre Depot (synthetic),11113.91,Multi-Modal Air-to-Land,11.54,13.47,279,5.6,1.78,0.673,12.37,Deep Frozen / Dry Ice,29990.67,Routine Restock
3,SC-10003,Busan Pharmaceutical Logistics Depot (synthetic),Hamburg Medical Centre Depot (synthetic),9180.55,Multi-Modal Air-to-Land,53.13,36.07,732,0.8,3.20,0.832,36.94,-20°C,49255.63,Critical Emergency / Direct Patient Treatment
4,SC-10004,Busan Pharmaceutical Logistics Depot (synthetic),Singapore Medical Centre Depot (synthetic),2762.27,Air,0.50,10.98,3800,1.9,2.77,0.741,67.53,2°C to 8°C,42010.69,Routine Restock


### What each column means

| Column | Meaning | Unit |
|---|---|---|
| `Shipment_ID` | Unique shipment code | text |
| `Place_of_Origin` | Depot the shipment leaves from | text |
| `Place_of_Destination` | Hospital depot it is going to | text |
| `Distance_km` | Length of the route | km |
| `Transport_Mode` | Air, Land, or Multi-Modal Air-to-Land | category |
| `Lead_Time_Days` | Agreed delivery window | days |
| `Transit_Duration_Hours` | Expected door-to-door travel time | hours |
| `Stock_Volume` | Units in the shipment | count |
| `Geopolitical_Risk_Score` | Risk along the route | 0 to 10 |
| `Fuel_Price_Index` | Fuel cost level | index |
| `Carrier_Reliability_Score` | How dependable the carrier is | 0 to 1 |
| `Buffer_of_Package` | How long the packaging holds its temperature | hours |
| `Storage_standard` | Required temperature range | category |
| `Cargo_Value_USD` | Value of the cargo | USD |
| `Hospital_Urgency_Tier` | How badly the hospital needs it | category |

## 2. Is anything missing or duplicated

For each column: its type, how many values are missing, and how many different values it has.

In [3]:
overview = pd.DataFrame({
    "type": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique_values": raw.nunique(),
    "example": raw.iloc[0].astype(str),
})
print(f"Missing values in total : {int(raw.isna().sum().sum())}")
print(f"Fully duplicated rows   : {int(raw.duplicated().sum())}")
print(f"Shipment_ID is unique   : {raw['Shipment_ID'].is_unique}")
overview

Missing values in total : 0
Fully duplicated rows   : 0
Shipment_ID is unique   : True


,type,missing,unique_values,example
Shipment_ID,str,0,5000,SC-10000
Place_of_Origin,str,0,8,Singapore Pharmaceutical Logistics Depot (synt...
Place_of_Destination,str,0,9,Los Angeles Medical Centre Depot (synthetic)
Distance_km,float64,0,4998,5930.83
Transport_Mode,str,0,3,Multi-Modal Air-to-Land
Lead_Time_Days,float64,0,2601,41.39
Transit_Duration_Hours,float64,0,3402,34.39
Stock_Volume,int64,0,3090,6241
Geopolitical_Risk_Score,float64,0,101,5.0
Fuel_Price_Index,float64,0,331,2.43


**What this tells us:** the file is complete. No blanks, no duplicate rows, every shipment has its own ID.

A perfectly complete file is itself a sign that the data is generated, since real logistics records always have gaps. The app must still cope with missing input, so we handle that in the code rather than rely on the data being tidy.

## 3. What the numbers and categories look like

### Numeric columns

In [4]:
numeric_cols = raw.select_dtypes("number").columns.tolist()
raw[numeric_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Distance_km,5000.0,7704.06,4199.69,500.17,4036.01,7750.12,11347.46,14995.91
Lead_Time_Days,5000.0,19.36,31.41,0.50,2.11,8.25,21.21,236.39
Transit_Duration_Hours,5000.0,31.05,34.37,0.76,12.58,24.13,37.83,399.77
Stock_Volume,5000.0,2601.77,2515.35,250.00,621.00,1535.50,3948.00,9988.00
Geopolitical_Risk_Score,5000.0,5.08,2.88,0.00,2.60,5.10,7.50,10.00
Fuel_Price_Index,5000.0,2.85,0.96,1.20,2.02,2.84,3.71,4.50
Carrier_Reliability_Score,5000.0,0.75,0.14,0.50,0.63,0.76,0.88,1.00
Buffer_of_Package,5000.0,52.01,32.73,0.54,21.69,47.13,79.30,119.92
Cargo_Value_USD,5000.0,70476.29,31780.41,15016.30,42455.22,71562.90,97870.41,124995.99


In [5]:
fig = make_subplots(rows=3, cols=3, subplot_titles=numeric_cols,
                    vertical_spacing=0.14, horizontal_spacing=0.07)
for i, col in enumerate(numeric_cols):
    counts, edges = np.histogram(raw[col], bins=40)  # 40 equal-width bins
    centers = (edges[:-1] + edges[1:]) / 2
    fig.add_trace(
        go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.9, marker_color=BLUE,
               hovertemplate=col + ": %{x:,.2f}<br>Shipments: %{y}<extra></extra>"),
        row=i // 3 + 1, col=i % 3 + 1,
    )
fig.update_annotations(font=dict(size=12, color=INK))
style(fig, "How each numeric column is spread (count of shipments)", height=720)
fig.update_layout(showlegend=False)
fig.show()

**What this tells us:**

- `Geopolitical_Risk_Score`, `Fuel_Price_Index`, `Carrier_Reliability_Score`, `Cargo_Value_USD` and `Distance_km` are almost flat. Every value is about equally likely, which is how a random number generator behaves and not how real routes behave.
- `Lead_Time_Days` and `Transit_Duration_Hours` have a long tail. A few shipments have a delivery window of more than 200 days, which is not believable for temperature-sensitive medicine.
- `Buffer_of_Package` has five repeating peaks that end at 24, 48, 72, 96 and 120 hours. That looks like five standard packaging sizes. We come back to this in Check D.

### Category columns

In [6]:
category_cols = ["Transport_Mode", "Storage_standard", "Hospital_Urgency_Tier"]

fig = make_subplots(rows=3, cols=1, subplot_titles=category_cols, vertical_spacing=0.16)
for i, col in enumerate(category_cols, start=1):
    counts = raw[col].value_counts().sort_values()
    share = counts / len(raw)
    fig.add_trace(
        go.Bar(x=counts.values, y=counts.index, orientation="h", marker_color=BLUE,
               text=[f"{c:,}  ({s:.0%})" for c, s in zip(counts, share)],
               textposition="outside", textfont=dict(color=INK), cliponaxis=False,
               hovertemplate="%{y}<br>Shipments: %{x:,}<extra></extra>"),
        row=i, col=1,
    )
fig.update_annotations(font=dict(size=12, color=INK), x=0, xanchor="left")
fig.update_xaxes(range=[0, raw[category_cols].apply(lambda s: s.value_counts().max()).max() * 1.25])
style(fig, "How many shipments fall in each category", height=620)
fig.update_layout(showlegend=False, bargap=0.35, margin=dict(l=300))
fig.show()

In [7]:
def city(place):
    """'Los Angeles Medical Centre Depot (synthetic)' -> 'Los Angeles'"""
    return place.str.replace(
        r"\s+(Pharmaceutical Logistics|Medical Centre) Depot.*$", "", regex=True
    )

cities = pd.DataFrame({
    "shipments_from": city(raw["Place_of_Origin"]).value_counts(),
    "shipments_to": city(raw["Place_of_Destination"]).value_counts(),
}).fillna(0).astype(int).sort_values("shipments_to", ascending=False)
print(f"Origin cities: {raw['Place_of_Origin'].nunique()}, destination cities: {raw['Place_of_Destination'].nunique()}")
cities

Origin cities: 8, destination cities: 9


,shipments_from,shipments_to
Marseille,0,583
Rotterdam,646,574
Busan,667,570
Shanghai,644,561
Dubai,610,559
Antwerp,635,552
Los Angeles,602,537
Hamburg,597,536
Singapore,599,528


**What this tells us:**

- Most shipments are multi-modal (air then road). Land-only is the smallest group.
- About 6 in 10 need 2°C to 8°C storage, the standard range for vaccines and biologics.
- About 15% are tagged as the most urgent tier. These are the ones a coordinator must protect first.
- Nine cities are in the network. Marseille only receives shipments, it never sends any.
- Place names end with "(synthetic)" and one urgency label is very long. Both need tidying before they go on screen.

## 4. Do the values make physical sense

This is the most important section. A model learns whatever patterns are in the data. If the data breaks the laws of physics, the model learns nonsense.

We run four checks. Three fail and one passes.

### Check A: does the same route always have the same distance?

The distance between two cities does not change from one shipment to the next.

In [8]:
checks = raw.copy()
checks["origin_city"] = city(checks["Place_of_Origin"])
checks["dest_city"] = city(checks["Place_of_Destination"])

routes = (
    checks.groupby(["origin_city", "dest_city"])["Distance_km"]
    .agg(shipments="count", shortest_km="min", longest_km="max")
    .round(0)
)
routes["longest_vs_shortest"] = (routes["longest_km"] / routes["shortest_km"]).round(1)
inconsistent_routes = int((routes["longest_vs_shortest"] > 1.5).sum())

print(f"Routes in the file: {len(routes)}")
print(f"Routes where the longest distance is over 1.5x the shortest: {inconsistent_routes}")
routes.sort_values("longest_vs_shortest", ascending=False).head(6)

Routes in the file: 64
Routes where the longest distance is over 1.5x the shortest: 64


,,shipments,shortest_km,longest_km,longest_vs_shortest
origin_city,dest_city,,,,
Busan,Antwerp,85,500.0,14994.0,30.0
Dubai,Antwerp,76,502.0,14938.0,29.8
Rotterdam,Antwerp,83,500.0,14854.0,29.7
Antwerp,Rotterdam,97,503.0,14865.0,29.6
Los Angeles,Marseille,72,509.0,14892.0,29.3
Busan,Shanghai,91,514.0,14984.0,29.2


**Result: fail.** The same pair of cities shows up with distances from a few hundred km to over 14,000 km. `Distance_km` was drawn at random and is not tied to the route.

### Check B: are the travel speeds possible?

Speed is distance divided by travel time. A cargo jet cruises at about 900 km/h. A truck averages about 90 km/h at best.

In [9]:
checks["speed_kmh"] = checks["Distance_km"] / checks["Transit_Duration_Hours"]
JET_KMH, TRUCK_KMH = 900, 90

mode_colors = {"Air": BLUE, "Multi-Modal Air-to-Land": ORANGE, "Land": AQUA}
fig = go.Figure()
for mode, color in mode_colors.items():
    fig.add_trace(go.Box(
        y=checks.loc[checks["Transport_Mode"] == mode, "speed_kmh"],
        name=mode, marker_color=color, line_width=2, boxpoints="outliers",
        marker=dict(size=4, opacity=0.5),
    ))
for limit, label in [(JET_KMH, "Cargo jet cruise, about 900 km/h"),
                     (TRUCK_KMH, "Truck on a highway, about 90 km/h")]:
    fig.add_hline(y=limit, line=dict(color=INK, width=1, dash="dash"))
    # on a log axis, label positions are given as log10 of the value
    fig.add_annotation(xref="paper", x=0.01, y=np.log10(limit), text=label, showarrow=False,
                       xanchor="left", yanchor="bottom", font=dict(color=INK))
fig.update_yaxes(type="log", range=[1, 4], tickvals=[10, 30, 100, 300, 1000, 3000, 10000], title_text="Implied speed (km/h, log scale)")
style(fig, "Implied door-to-door speed by transport mode", height=480)
fig.update_layout(showlegend=False)
fig.show()

too_fast_air = (checks["Transport_Mode"] != "Land") & (checks["speed_kmh"] > JET_KMH)
too_fast_land = (checks["Transport_Mode"] == "Land") & (checks["speed_kmh"] > TRUCK_KMH)
print(f"Air or multi-modal shipments faster than a jet : {too_fast_air.sum():,}")
print(f"Land shipments faster than a truck             : {too_fast_land.sum():,}")
print(f"Fastest shipment in the file                   : {checks['speed_kmh'].max():,.0f} km/h")

Air or multi-modal shipments faster than a jet : 631
Land shipments faster than a truck             : 119
Fastest shipment in the file                   : 7,052 km/h


**Result: fail.** Hundreds of air shipments travel faster than any aircraft, and about a third of the land shipments are faster than a truck can drive. Distance and travel time were generated separately.

### Check C: can a truck actually drive that route?

Los Angeles is separated from every other city in the file by an ocean. A land-only shipment to or from Los Angeles is impossible. In this network only the four European cities (Rotterdam, Antwerp, Hamburg, Marseille) are realistically joined by road.

In [10]:
EUROPE = {"Rotterdam", "Antwerp", "Hamburg", "Marseille"}

is_land = checks["Transport_Mode"] == "Land"
touches_la = (checks["origin_city"] == "Los Angeles") | (checks["dest_city"] == "Los Angeles")
inside_europe = checks["origin_city"].isin(EUROPE) & checks["dest_city"].isin(EUROPE)

land_across_ocean = is_land & touches_la
land_outside_europe = is_land & ~inside_europe

print(f"Land shipments in total                       : {is_land.sum():,}")
print(f"Land shipments to or from Los Angeles (ocean) : {land_across_ocean.sum():,}")
print(f"Land shipments not between European cities    : {land_outside_europe.sum():,}")
checks.loc[is_land].groupby(["origin_city", "dest_city"]).size().rename("land_shipments").sort_values(ascending=False).head(6).to_frame()

Land shipments in total                       : 333
Land shipments to or from Los Angeles (ocean) : 0
Land shipments not between European cities    : 0


,,land_shipments
origin_city,dest_city,
Rotterdam,Antwerp,47
Antwerp,Rotterdam,43
Hamburg,Rotterdam,40
Rotterdam,Hamburg,39
Antwerp,Hamburg,36
Hamburg,Antwerp,35


**Result: pass.** Every land-only shipment runs between two European cities, so the transport mode does respect geography. This column can be trusted. The distances on those road routes are still wrong (see Check A).

### Check D: do the time columns agree with each other?

Three columns are measured in time: the delivery window (`Lead_Time_Days`), the travel time (`Transit_Duration_Hours`) and the packaging hold time (`Buffer_of_Package`). In a planned shipment the window is longer than the trip, and the packaging is chosen to last the whole trip.

**Delivery window first.**

In [11]:
window_hours = checks["Lead_Time_Days"] * 24
window_too_short = window_hours < checks["Transit_Duration_Hours"]
window_too_long = checks["Lead_Time_Days"] > 30

print(f"Delivery window shorter than the trip itself : {window_too_short.sum():,} ({window_too_short.mean():.0%})")
print(f"Delivery window longer than 30 days          : {window_too_long.sum():,} ({window_too_long.mean():.0%})")
print(f"Longest delivery window                      : {checks['Lead_Time_Days'].max():.0f} days")

Delivery window shorter than the trip itself : 516 (10%)
Delivery window longer than 30 days          : 877 (18%)
Longest delivery window                      : 236 days


**Result: fail.** One shipment in ten is promised sooner than it can physically arrive, and almost one in five has a window of over a month, which no cold-chain medicine would be given.

**Now the packaging.** Each dot below is one shipment.

In [12]:
buffer_too_short = checks["Buffer_of_Package"] < checks["Transit_Duration_Hours"]

X_MAX = 140
fig = go.Figure()
for mask, name, color in [
    (~buffer_too_short, "Buffer longer than the trip", BLUE),
    (buffer_too_short, "Buffer shorter than the trip", ORANGE),
]:
    fig.add_trace(go.Scatter(
        x=checks.loc[mask, "Transit_Duration_Hours"], y=checks.loc[mask, "Buffer_of_Package"],
        mode="markers", name=f"{name} ({mask.sum():,})",
        marker=dict(color=color, size=5, opacity=0.55),
        hovertemplate="Travel time: %{x} h<br>Buffer: %{y} h<extra></extra>",
    ))
fig.add_trace(go.Scatter(x=[0, X_MAX], y=[0, X_MAX], mode="lines", name="Buffer = travel time",
                         line=dict(color=INK, width=2, dash="dash"), hoverinfo="skip"))
fig.update_xaxes(range=[0, X_MAX], title_text="Transit_Duration_Hours (travel time)")
fig.update_yaxes(range=[0, 125], title_text="Buffer_of_Package (hours)")
style(fig, "Packaging buffer against travel time", height=520)
fig.update_layout(legend=dict(orientation="h", y=1.08, x=0, font=dict(color=INK)))
fig.show()

print(f"Shipments off the right edge of the chart (travel time over {X_MAX} h): "
      f"{(checks['Transit_Duration_Hours'] > X_MAX).sum():,}")

Shipments off the right edge of the chart (travel time over 140 h): 94


The dots are not random. They form five slanted bands that start at 24, 48, 72, 96 and 120 hours and slope downward as travel time grows. The next cell tests what that pattern means.

In [13]:
PACK_CLASSES = np.array([24, 48, 72, 96, 120])  # the five band tops seen in the chart

# Round each buffer up to the band it sits in, then see how much has been "used"
position = np.searchsorted(PACK_CLASSES, checks["Buffer_of_Package"]).clip(max=len(PACK_CLASSES) - 1)
checks["pack_class_hours"] = PACK_CLASSES[position]
checks["hours_used"] = checks["pack_class_hours"] - checks["Buffer_of_Package"]

used_fits_trip = checks["hours_used"] <= checks["Transit_Duration_Hours"] + 0.01
print(f"Shipments where the hours used are no more than the travel time: {used_fits_trip.mean():.0%}")
print(f"Typical share of the trip already used: {(checks['hours_used'] / checks['Transit_Duration_Hours']).median():.0%}")
checks["pack_class_hours"].value_counts().sort_index().rename("shipments").to_frame()

Shipments where the hours used are no more than the travel time: 100%
Typical share of the trip already used: 41%


,shipments
pack_class_hours,
24,1432
48,1116
72,1050
96,950
120,452


**Result: cannot be judged from the raw file.**

The pattern fits one explanation for every shipment in the file: the raw `Buffer_of_Package` is the hold time **left right now**. It is a packaging size (24, 48, 72, 96 or 120 hours) minus the hours the shipment has already been travelling.

That is a useful number, but the raw file does not say how far along each shipment is. So for the orange dots we cannot tell whether the packaging will really run out, or whether the shipment is nearly there. A coordinator needs both halves: how much buffer is left, and how much trip is left.

The rebuilt file fixes this by giving each column one clear meaning: `Buffer_of_Package` is the hold time counted from dispatch, and `Transit_Duration_Hours` is the expected door-to-door time.

### Scorecard for all four checks

In [14]:
n = len(checks)
scorecard = pd.DataFrame(
    [
        ("A", "Route has inconsistent distances", f"{inconsistent_routes} of {len(routes)} routes", inconsistent_routes / len(routes), "fail"),
        ("B", "Air or multi-modal faster than a jet", f"{too_fast_air.sum():,} shipments", too_fast_air.sum() / (~is_land).sum(), "fail"),
        ("B", "Land faster than a truck", f"{too_fast_land.sum():,} shipments", too_fast_land.sum() / is_land.sum(), "fail"),
        ("C", "Land route that is not drivable", f"{land_outside_europe.sum():,} shipments", land_outside_europe.sum() / is_land.sum(), "pass"),
        ("D", "Delivery window shorter than the trip", f"{window_too_short.sum():,} shipments", window_too_short.sum() / n, "fail"),
        ("D", "Delivery window longer than 30 days", f"{window_too_long.sum():,} shipments", window_too_long.sum() / n, "fail"),
        ("D", "Buffer left is shorter than the whole trip", f"{buffer_too_short.sum():,} shipments", buffer_too_short.sum() / n, "cannot judge"),
    ],
    columns=["check", "what_we_looked_for", "how_many", "share_of_group", "verdict"],
)
scorecard["share_of_group"] = scorecard["share_of_group"].map("{:.0%}".format)
scorecard

,check,what_we_looked_for,how_many,share_of_group,verdict
0,A,Route has inconsistent distances,64 of 64 routes,100%,fail
1,B,Air or multi-modal faster than a jet,631 shipments,14%,fail
2,B,Land faster than a truck,119 shipments,36%,fail
3,C,Land route that is not drivable,0 shipments,0%,pass
4,D,Delivery window shorter than the trip,516 shipments,10%,fail
5,D,Delivery window longer than 30 days,877 shipments,18%,fail
6,D,Buffer left is shorter than the whole trip,"1,486 shipments",30%,cannot judge


## 5. Do the columns relate to each other

In real shipments, longer routes take longer, riskier routes are slower, and unreliable carriers run late. A correlation of 0 means two columns have nothing to do with each other. A value near +1 or -1 means they move together.

In [15]:
corr = raw[numeric_cols].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1,
    colorscale=[[0, "#2a78d6"], [0.5, "#f0efec"], [1, "#e34948"]],
    text=corr.values, texttemplate="%{text:.2f}", textfont=dict(color=INK, size=11),
    xgap=2, ygap=2, colorbar=dict(title="Correlation", thickness=12),
    hovertemplate="%{y}<br>%{x}<br>Correlation: %{z:.2f}<extra></extra>",
))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=-35)
style(fig, "Correlation between numeric columns", height=600)
fig.update_layout(margin=dict(l=190, b=150))
fig.show()

off_diagonal = corr.where(~np.eye(len(corr), dtype=bool)).abs().stack().dropna()
print(f"Strongest link between any two columns: {off_diagonal.max():.2f}")
print(f"Pairs with a link stronger than 0.30  : {int((off_diagonal > 0.30).sum() / 2)} of {len(off_diagonal) // 2}")

Strongest link between any two columns: 0.34
Pairs with a link stronger than 0.30  : 1 of 36


**What this tells us:** almost every cell is close to zero. Carrier reliability, geopolitical risk and fuel price are not linked to travel time at all. The columns were generated independently, so there is no cause-and-effect for a model to learn from.

## 6. Is there an outcome to predict

A failure-prediction model needs a column that says what actually happened to each shipment.

In [16]:
outcome_words = ["fail", "late", "delay", "excursion", "damage", "spoil", "outcome", "status"]
outcome_like = [c for c in raw.columns if any(w in c.lower() for w in outcome_words)]
print("Columns that record what happened to the shipment:", outcome_like or "none")

Columns that record what happened to the shipment: none


**Result:** there is none. The raw file describes shipments but never says which ones failed. Without that, nothing can be trained and nothing can be scored.

## 7. What we found and what happens next

### Findings

| # | Finding | Why it matters |
|---|---|---|
| 1 | 5,000 shipments, 15 columns, no missing values, no duplicates | The file is easy to load, but too tidy to be real |
| 2 | Distance is not tied to the route | The model cannot learn that long routes are riskier |
| 3 | Travel speeds are impossible for many shipments | Distance and travel time contradict each other |
| 4 | Land shipments only run between European cities | Transport mode respects geography, so we keep it |
| 5 | Many delivery windows are shorter than the trip, or longer than a month | Late delivery is one of the failures Aura predicts, so the window must be believable |
| 6 | The buffer is hold time left, in five packaging sizes, but trip progress is not recorded | We cannot tell from the raw file whether the packaging will last |
| 7 | Columns are almost unrelated to each other | There is no signal linking risk factors to delays |
| 8 | No outcome column | There is nothing to predict yet |

### Decision

The raw file cannot be used for training as it is. We keep the columns that are plausible on their own and rebuild the four that contradict each other or cannot be interpreted.

| Keep as given | Rebuild from the route | Add |
|---|---|---|
| `Shipment_ID`, origin, destination | `Distance_km` (real distance between the two cities) | `Planned_Transit_Hours` |
| `Transport_Mode` (very short European hops switched to road) | `Lead_Time_Days` (window sized to the trip and urgency) | `Handover_Count` |
| `Stock_Volume`, `Cargo_Value_USD` | `Transit_Duration_Hours` (plan plus known delay) | Simulated outcomes: `Late_Delivery`, `Temperature_Excursion`, `Handling_Damage` |
| `Geopolitical_Risk_Score`, `Fuel_Price_Index`, `Carrier_Reliability_Score` | `Buffer_of_Package` (hold time from dispatch, same five packaging sizes) | Target: `Shipment_Failure` |
| `Storage_standard`, `Hospital_Urgency_Tier` | | |

This is done by `python src/build_dataset.py`, which writes `data/processed/aura_shipments.csv`.

### Next

Notebook 02 will check the rebuilt file with the same tests used here, then explore what separates failed shipments from safe ones.

### Limits we state openly

- Routes, schedules and outcomes are simulated. Results show that the method works, not how it would score on a real carrier's records.
- The four outcome columns are only known after delivery. They are never used as model inputs.